## ContainerEvent End-to-End Test

This notebook validates the ContainerEvent feature which treats the full measurement container (start_ts to stop_ts) as a single event instance without requiring a time-series expression.

### General Setup

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from mda_reporting.config.config_parser import (
    MdaConfig,
    MeasurementDimensions,
    QueryEngine,
    Solvers,
    Source,
    UnitySink,
)
from mda_reporting.core.report import Report
from mda_reporting.events.container_event import ContainerEvent
from mda_reporting.events.basic_event import BasicEvent

### Initialization

### Test 2: ContainerEvent with BasicEvent

In [0]:
# Create configuration for mixed events
config_mixed = {
      "incremental":{
      "enabled": True

  },
  "source": {
    "container_metrics_table": "avl_databricks_mvp.silver.container_metric",
    "channel_metrics_table": "avl_databricks_mvp.silver.channel_metric",
    "channels_uri": "avl_databricks_mvp.silver.channel_data"
  },
  "unity_sink": {
    "catalog": "development",
    "schema": "gold_e2e",
    "table_prefix": "container_events"
  },
    "query_engine": {"solver": "BasicNarrowSolver"},
    "measurement_dimensions": [
    "container_id",
    "start_ts",
    "stop_ts",
  ]
}

for tbl in spark.catalog.listTables(f'{config_mixed["unity_sink"]["catalog"]}.{config_mixed["unity_sink"]["schema"]}'):
  spark.sql(f'DROP TABLE IF EXISTS {config_mixed["unity_sink"]["catalog"]}.{config_mixed["unity_sink"]["schema"]}.{tbl.name}')

report_mixed = Report(name="mixed_events_report", spark=spark, config=config_mixed)
db_mixed = report_mixed.get_db()

In [0]:
# Add ContainerEvent
container_evt_mixed = ContainerEvent(
    name="container_duration",
    desc="Full container duration"
)
report_mixed.add_event(container_evt_mixed)

# Add BasicEvent
eng_rpm = db_mixed.query.channel(channel_name="is1_eng_speed", data_key="TM")
basic_evt = BasicEvent(
    name="rpm_event",
    expr=eng_rpm > -1,
    desc="Engine RPM > -1",
    required_channels=["is1_eng_speed"]
)
report_mixed.add_event(basic_evt)

print(f"Added ContainerEvent: {container_evt_mixed.name}")
print(f"Added BasicEvent: {basic_evt.name}")

In [0]:
# Determine mixed report
print("Determining mixed events report...")
report_mixed.determine_report()
print("Report determination complete")

### Verify Mixed Event Results

In [0]:
# Check both event types are present
mixed_event_dfs = report_mixed.event_dfs

print("Event types in mixed report:", list(mixed_event_dfs.keys()))
assert "CONTAINER_EVENT" in mixed_event_dfs
assert "BASIC_EVENT" in mixed_event_dfs
print("✓ Both event types present")

In [0]:
# Show ContainerEvent instances
print("=== ContainerEvent Instances ===")
container_mixed_df = mixed_event_dfs["CONTAINER_EVENT"]["changed"]
display(container_mixed_df.orderBy("container_id"))

container_rows = container_mixed_df.collect()
print(f"Total ContainerEvent instances: {len(container_rows)}")

for row in container_rows:
    assert row.event_instance_id == -1
print("✓ All ContainerEvent instances have event_instance_id = -1")

In [0]:
from pyspark.sql import functions as F

In [0]:
# Show BasicEvent instances
print("=== BasicEvent Instances ===")
basic_mixed_df = mixed_event_dfs["BASIC_EVENT"]["changed"]
display(basic_mixed_df.orderBy("container_id", "start_ts"))

basic_rows = basic_mixed_df.where(F.col('event_instance_id') == -1)
# print(f"Total BasicEvent instances: {len(basic_rows)}")

assert basic_rows.count() == 0
print("✓ All BasicEvent instances have non-negative event_instance_id (CRC32 hash)")

In [0]:
# Verify event metadata for both types
mixed_metadata = report_mixed.event_metadata_dfs

print("=== ContainerEvent Metadata ===")
display(mixed_metadata["CONTAINER_EVENT"])

print("=== BasicEvent Metadata ===")
display(mixed_metadata["BASIC_EVENT"])

container_dim = mixed_metadata["CONTAINER_EVENT"].collect()
basic_dim = mixed_metadata["BASIC_EVENT"].collect()

assert len(container_dim) == 1
assert len(basic_dim) == 1
assert container_dim[0].event_name == "container_duration"
assert container_dim[0].event_expression == "NA"
assert basic_dim[0].event_name == "rpm_event"
assert basic_dim[0].event_expression != "NA"
assert container_dim[0].event_type == "CONTAINER_EVENT"
assert basic_dim[0].event_type == "BASIC_EVENT"


print("✓ Both event definitions stored correctly")

### Optional: Persist Results

In [0]:
# Uncomment to persist results to Unity Catalog
# report_standalone.persist_results()
report_mixed.persist_results()
print("✓ Results persisted*")

### Summary

**Test Results:**
- Standalone ContainerEvent: ✅ event_instance_id=-1, timestamps match containers
- Mixed Events: ✅ Both ContainerEvent and BasicEvent coexist correctly
- Metadata: ✅ Both event definitions stored with correct properties

**Key Features Validated:**
1. ContainerEvent derives boundaries from container metadata (no expression required)
2. Uses sentinel value (-1) for event_instance_id
3. Compatible with BasicEvent in the same report
4. Both event types write to shared output tables